In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from classical_estimates import *
from kll import kll
from scipy.ndimage import gaussian_filter
from processing import *

In [2]:
folder = 'temp/'
files = sorted(glob.glob(folder + '*.fits'))
files

['temp/phi-fdt-ilam_20260310T040003_V202609031907C_0663100100.fits',
 'temp/phi-fdt-ilam_20260310T040405_V202609031908C_0663100125.fits',
 'temp/phi-fdt-ilam_20260310T040805_V202609031909C_0663100150.fits',
 'temp/phi-fdt-ilam_20260310T041205_V202609031910C_0663100175.fits',
 'temp/phi-fdt-ilam_20260310T041605_V202609031911C_0663100200.fits',
 'temp/phi-fdt-ilam_20260310T042005_V202609031912C_0663100225.fits',
 'temp/phi-fdt-ilam_20260310T042405_V202609031913C_0663100250.fits',
 'temp/phi-fdt-ilam_20260310T042805_V202609031914C_0663100275.fits',
 'temp/phi-fdt-ilam_20260310T043205_V202609031915C_0663100300.fits']

In [3]:
vlcps = []
vrcps = []
weights = []

centers = []

for file in files:
    with fits.open(file) as hdul:
        data = hdul[0].data
        header = hdul[0].header

    contpos = header['CONTPOS'] - 1

    #data = rebin(data, 8, update_header=header)

    lcp = (data[:,0] + data[:,3]) / 2
    rcp = (data[:,0] - data[:,3]) / 2

    vlcp = get_wv_shift(lcp, header)
    vrcp = get_wv_shift(rcp, header)

    xc = header['CRPIX2'] - 1
    yc = header['CRPIX1'] - 1

    vlcps += [vlcp]
    vrcps += [vrcp]
    weights += [data[contpos,0]]
    centers += [(xc, yc)]

vlcps = np.array(vlcps)
vrcps = np.array(vrcps)
weights = np.array(weights)
centers = np.array(centers)

ql = kll(np.nan_to_num(vlcps), centers, weights=np.nan_to_num(weights), niter=1000, sigma=1e-3, vmin=-0.2, vmax=0.2)
qr = kll(np.nan_to_num(vrcps), centers, weights=np.nan_to_num(weights), niter=1000, sigma=1e-3, vmin=-0.2, vmax=0.2)

q = ql - qr
np.savez('bias.npz', bias=q)



In [15]:
plt.figure(figsize=(10,10))
plt.imshow(ql, 'seismic', vmin=-0.05, vmax=0.05)
plt.tight_layout()

In [22]:
q = ql - qr
q -= np.nanmedian(q[~np.isnan(vlcps[0])])
q = gaussian_filter(q, 50)

In [23]:
plt.figure(figsize=(10,10))
plt.imshow(q, 'seismic', vmin=-3e-4, vmax=3e-4)
plt.tight_layout()

In [24]:
np.savez('/home/ulyanov/data/solo/bias.npz', bias=q)